In [370]:
import pandas as pd
import os
import glob as glob
from pathlib import Path

In [371]:
def all():
    path = Path("/Users/danielcm/Desktop/SickKids/Anvio/07_PANGENOMICS/")
    for dir in path.glob("*-PROJECT"):
        print(f"PROCESSING PROJECT {dir.name}")
        genome1 = glob.glob(os.path.join(path,dir,"*.txt"))[0]
        misc_data = dir / "SUMMARIZE/misc_data_layers/"
        f_in1 = pd.read_csv(os.path.join(path,genome1), sep="\t")
        f_in2 = pd.read_csv(os.path.join(path,dir,misc_data, "default.txt"), sep="\t")
        f_in3 = pd.read_csv(os.path.join(path,dir,"SUMMARIZE/gene_clusters_summary.txt"), sep="\t")
        num_genomes = len(f_in1)
        df_pangenome = getting_pangenome(f_in3, num_genomes)
        df_genomes, df_file2 = get_genome_ids(f_in1, f_in2)
        df_final = pd.merge(df_genomes, df_pangenome, on="ID", how="outer")
        df_final = pd.merge(df_final, df_file2, on="ID", how="outer")
        df_final.to_csv(os.path.join(path,dir,"final_summary.csv"), index=False)
    
    i = 0
    j = 0
    for dir in path.glob("*-PROJECT"):
        if i == 0:
            df1 = pd.read_csv(os.path.join(path,dir,"final_summary.csv"))
            i = i +1
            if j == 1:
                df_final = pd.concat([df_final, df1], axis=0, join="inner")
                i = 0
        elif i == 1:
            df2 = pd.read_csv(os.path.join(path,dir,"final_summary.csv"))
            df_final = pd.concat([df1, df2], axis=0, join="inner")
            j = j + 1
            i = 0

    print(df_final)    
    
    return()

In [372]:
def get_genome_ids(file_in, file_in2):
    file_in["contigs_db_path"] = file_in["contigs_db_path"].apply(lambda x: x.split("/")[-1])
    file_in["contigs_db_path"] = file_in["contigs_db_path"].apply(lambda x: x.split("_")[1])
    print(file_in[["name","contigs_db_path"]])
    df_genomes = file_in[["name","contigs_db_path"]].rename(columns={"name": "ID", "contigs_db_path": "Consortium"})
    df_file2 = file_in2.rename(columns={"layers": "ID"})
    print(df_genomes)
    return df_genomes, df_file2

In [373]:
def getting_pangenome(file_in, num_genomes):
    df_cluster = file_in.drop_duplicates(subset=["gene_cluster_id"])
    num_unique_gcs = len(df_cluster)
    core_pangenome = df_cluster[df_cluster["num_genomes_gene_cluster_has_hits"] == num_genomes]
    core_pangenome_size = len(core_pangenome)
    records = []

    for genome_id in sorted(file_in["genome_name"].unique()):
        df_specific_genome = file_in[file_in["genome_name"] == genome_id]
        df_specific_genome = df_specific_genome.drop_duplicates(subset=["gene_cluster_id"])
        accessory_genome = df_specific_genome[(df_specific_genome["num_genomes_gene_cluster_has_hits"] < num_genomes) & 
                                            (df_specific_genome["num_genomes_gene_cluster_has_hits"] > 1)]
        strain_specific_genome = df_specific_genome[df_specific_genome["num_genomes_gene_cluster_has_hits"] == 1]
        records.append({"ID": genome_id, 
                        "core_pangenome_size": core_pangenome_size, 
                        "accessory_genome_size": len(accessory_genome), 
                        "strain_specific_genome_size": len(strain_specific_genome)})
    return(pd.DataFrame(records))


In [374]:
all()

PROCESSING PROJECT PVULGATUS-PROJECT
     name contigs_db_path
0  Bv_065              S2
1  Bv_007             NS1
2  Bv_037             NS6
3  Bv_093              S5
       ID Consortium
0  Bv_065         S2
1  Bv_007        NS1
2  Bv_037        NS6
3  Bv_093         S5
PROCESSING PROJECT AONDERDONKII-PROJECT
     name contigs_db_path
0  Ao_003             NS1
1  Ao_031             NS6
       ID Consortium
0  Ao_003        NS1
1  Ao_031        NS6
PROCESSING PROJECT ELENTA-PROJECT
     name contigs_db_path
0  El_044             NS6
1  El_045             NS6
2  El_100              S5
3  El_072              S2
4  El_073              S2
5  El_014             NS1
       ID Consortium
0  El_044        NS6
1  El_045        NS6
2  El_100         S5
3  El_072         S2
4  El_073         S2
5  El_014        NS1
PROCESSING PROJECT BLONGUM-PROJECT
       name contigs_db_path
0  Blss_040             NS6
1  Blss_094              S5
2  Blss_067              S2
3    Ba_008             NS1
4  Blss_0

()